In [62]:
import torch
import torch.nn as nn
import numpy as np
from torch.nn.utils.rnn import pack_padded_sequence

np.set_printoptions(suppress=True, threshold=np.inf)
torch.set_printoptions(sci_mode=False)

In [63]:
if torch.xpu.is_available():
    device = "xpu"

elif torch.cuda.is_available():
    device ="cuda"

else:
    device = "cpu"

print(device)

xpu


In [64]:
n = 1200
dataset = np.arange(1, n + 1)

input_size = 1
sequence_length = 5
number_of_examples = len(dataset) - sequence_length

X_list = []
y_list = []

for i in range(number_of_examples):
    X = dataset[i : i + sequence_length]
    X_list.append(X)
    y = dataset[i + sequence_length]
    y_list.append(y)

X_list = np.array(X_list)
y_list = np.array(y_list)

X_list = torch.from_numpy(X_list)
y_list = torch.from_numpy(y_list)

X_list = X_list.reshape(number_of_examples, sequence_length, input_size)
y_list = y_list.reshape(number_of_examples, input_size)

X_list = X_list.float()
y_list = y_list.float()

print("X shape:", X_list.shape)
print("y shape:", y_list.shape)
print("X dtype:", X_list.dtype)
print("y dtype:", y_list.dtype)

X shape: torch.Size([1195, 5, 1])
y shape: torch.Size([1195, 1])
X dtype: torch.float32
y dtype: torch.float32


In [65]:
class SequencePredictor(nn.Module):
    def __init__(self):
        super().__init__()
        self.rnn = nn.RNN(input_size=1, hidden_size=16, batch_first=True)
        self.fc = nn.Linear(16, 1)

    def forward(self, x):
        output, hidden = self.rnn(x)
        x = output[:, -1, :]
        x = self.fc(x)

        return x

In [66]:
model = SequencePredictor().to(device)

test_input = X_list[:4].to(device)

with torch.no_grad():
    prediction = model(test_input)

print("Input shape:", test_input.shape)
print("Prediction shape:", prediction.shape)
print("Predictions:", prediction)

Input shape: torch.Size([4, 5, 1])
Prediction shape: torch.Size([4, 1])
Predictions: tensor([[0.0489],
        [0.0338],
        [0.0188],
        [0.0052]], device='xpu:0')


In [67]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.001)

X_list = X_list.to(device)
y_list = y_list.to(device)

scale = 1200.0

X_norm = (X_list / scale).to(device)
y_norm = (y_list / scale).to(device)

In [68]:
epochs = 2000
for epoch in range(1, epochs + 1):
    model.train()
    model.zero_grad()
    predictions = model(X_norm)
    loss = criterion(predictions, y_norm)
    loss.backward()
    optimizer.step()

    if epoch % 100 == 0:
        print(f"Epoch: {epoch:>4}    |    Loss: {loss:.4f}")

Epoch:  100    |    Loss: 0.0086
Epoch:  200    |    Loss: 0.0002
Epoch:  300    |    Loss: 0.0002
Epoch:  400    |    Loss: 0.0002
Epoch:  500    |    Loss: 0.0001
Epoch:  600    |    Loss: 0.0001
Epoch:  700    |    Loss: 0.0001
Epoch:  800    |    Loss: 0.0001
Epoch:  900    |    Loss: 0.0000
Epoch: 1000    |    Loss: 0.0000
Epoch: 1100    |    Loss: 0.0000
Epoch: 1200    |    Loss: 0.0000
Epoch: 1300    |    Loss: 0.0000
Epoch: 1400    |    Loss: 0.0000
Epoch: 1500    |    Loss: 0.0000
Epoch: 1600    |    Loss: 0.0000
Epoch: 1700    |    Loss: 0.0000
Epoch: 1800    |    Loss: 0.0000
Epoch: 1900    |    Loss: 0.0000
Epoch: 2000    |    Loss: 0.0000


In [69]:
test_input = torch.Tensor([[[1091/1200], [1092/1200], [1093/1200], [1094/1200], [1095/1200]]]).to(device)

model.eval()
with torch.no_grad():
    prediction = model(test_input)

print(prediction * 1000)

tensor([[911.1298]], device='xpu:0')


## Text Modeling — Tokenization and Vocabulary

In [70]:
class nlp():

    def __init__(self, vocabulary_sentences):
        self.vocabulary_sentences = vocabulary_sentences
        self.vocabulary = self.build_vocabulary()

    def tokenize(self, x):
        return x.split()

    def build_vocabulary(self):
        vocabulary = {"<UNK>": 0, "<PAD>": 1}

        for sentence in self.vocabulary_sentences:
            words = self.tokenize(sentence)

            for word in words:
                if word not in vocabulary:
                    vocabulary[word] = len(vocabulary)

        return vocabulary

    def encode_sentence(self, sentence):
        words = self.tokenize(sentence)

        encoded_sentence = []

        for word in words:
            encoded_sentence.append(
                self.vocabulary.get(word, self.vocabulary["<UNK>"])
            )

        return encoded_sentence

    def pad_sequences(self, sentences):

        if not sentences:
            raise ValueError("No sentences provided.")

        max_len = max(len(sentence) for sentence in sentences)

        padded_sentences = []

        for sentence in sentences:
            padded_sentence = sentence.copy()

            while len(padded_sentence) < max_len:
                padded_sentence.append(self.vocabulary["<PAD>"])

            padded_sentences.append(padded_sentence)

        return padded_sentences

    def preprocess(self, sentences):

        encoded_sentences = [
            self.encode_sentence(sentence)
            for sentence in sentences
        ]

        lengths = torch.tensor(
            [len(sentence) for sentence in encoded_sentences],
            dtype=torch.long
        )

        padded_sentences = self.pad_sequences(encoded_sentences)

        input_ids = torch.tensor(
            padded_sentences,
            dtype=torch.long
        )

        return input_ids, lengths

In [71]:
vocabulary_sentences = [
    "I love this movie",
    "This movie is fantastic",
    "I really enjoyed this film",
    "This is a wonderful movie",
    "I hate this movie",
    "This movie is terrible",
    "I really disliked this film",
    "This is a horrible movie"
]

sentences = vocabulary_sentences.copy()

labels = [1, 1, 1, 1, 0, 0, 0, 0]

test_nlp = nlp(vocabulary_sentences)

input_ids, lengths = test_nlp.preprocess(sentences)

labels = torch.tensor(labels, dtype=torch.long)

print("Vocabulary:", test_nlp.vocabulary)
print("Input IDs shape:", input_ids.shape)
print("Lengths:", lengths)
print("Labels shape:", labels.shape)

Vocabulary: {'<UNK>': 0, '<PAD>': 1, 'I': 2, 'love': 3, 'this': 4, 'movie': 5, 'This': 6, 'is': 7, 'fantastic': 8, 'really': 9, 'enjoyed': 10, 'film': 11, 'a': 12, 'wonderful': 13, 'hate': 14, 'terrible': 15, 'disliked': 16, 'horrible': 17}
Input IDs shape: torch.Size([8, 5])
Lengths: tensor([4, 4, 5, 5, 4, 4, 5, 5])
Labels shape: torch.Size([8])


In [72]:
class SentimentClassifier(nn.Module):

    def __init__(self, vocab_size, embedding_dim, hidden_size, padding_idx):

        super().__init__()

        self.embedding = nn.Embedding(
            num_embeddings=vocab_size,
            embedding_dim=embedding_dim,
            padding_idx=padding_idx
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_size,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 2)

    def forward(self, input_ids, lengths):

        embedded = self.embedding(input_ids)

        packed = pack_padded_sequence(
            embedded,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False
        )

        _, (hidden, cell) = self.lstm(packed)

        final_hidden = hidden.squeeze(0)

        logits = self.fc(final_hidden)

        return logits

In [73]:
model = SentimentClassifier(
    vocab_size=len(test_nlp.vocabulary),
    embedding_dim=4,
    hidden_size=16,
    padding_idx=test_nlp.vocabulary["<PAD>"]
)

print(model)

SentimentClassifier(
  (embedding): Embedding(18, 4, padding_idx=1)
  (lstm): LSTM(4, 16, batch_first=True)
  (fc): Linear(in_features=16, out_features=2, bias=True)
)


In [74]:
logits = model(input_ids, lengths)

print("Logits:")
print(logits)

print("Logits shape:", logits.shape)

Logits:
tensor([[ 0.0126, -0.1776],
        [ 0.0857, -0.2031],
        [ 0.2269, -0.2111],
        [ 0.0476, -0.2596],
        [ 0.0989, -0.2329],
        [ 0.0634, -0.2651],
        [ 0.1664, -0.2085],
        [ 0.0493, -0.2179]], grad_fn=<AddmmBackward0>)
Logits shape: torch.Size([8, 2])


In [75]:
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(
    model.parameters(),
    lr=0.001
)

epochs = 200

for epoch in range(epochs):

    model.train()

    optimizer.zero_grad()

    logits = model(input_ids, lengths)

    loss = criterion(logits, labels)

    loss.backward()

    optimizer.step()

    if (epoch + 1) % 20 == 0:
        predictions = torch.argmax(logits, dim=1)
        accuracy = (predictions == labels).float().mean()

        print(
            f"Epoch [{epoch+1}/{epochs}] "
            f"Loss: {loss.item():.4f} "
            f"Accuracy: {accuracy.item()*100:.2f}%"
        )

Epoch [20/200] Loss: 0.6762 Accuracy: 62.50%
Epoch [40/200] Loss: 0.6509 Accuracy: 75.00%
Epoch [60/200] Loss: 0.6108 Accuracy: 87.50%
Epoch [80/200] Loss: 0.5377 Accuracy: 87.50%
Epoch [100/200] Loss: 0.4143 Accuracy: 100.00%
Epoch [120/200] Loss: 0.2570 Accuracy: 100.00%
Epoch [140/200] Loss: 0.1288 Accuracy: 100.00%
Epoch [160/200] Loss: 0.0641 Accuracy: 100.00%
Epoch [180/200] Loss: 0.0369 Accuracy: 100.00%
Epoch [200/200] Loss: 0.0242 Accuracy: 100.00%
